# Notebook 19 – Mini Assessment

**Dataset:** Online Retail transactions (`data.csv`)

**Structure:**
- **Part A – Concept Questions (Q1–Q17):** each question is followed by its answer.
- **Part B – Coding Exercises (E1–E9):** each exercise is followed by a solution.

> Tip: try each one yourself first, then read the answer/solution below it.

# Part A – Concept Questions

### Q1. Parameters vs Hyperparameters

**Answer:** **Parameters** are learned *from the data* during training (e.g., regression coefficients, tree split points). **Hyperparameters** are set *by us before training* and control how the model learns (e.g., `max_depth`, `C`, `n_estimators`, `learning_rate`).

### Q2. What is overfitting?

**Answer:** The model learns the training data too closely, including its noise, so it scores **high on training data but noticeably lower on validation/test data**. Fixes: more data, a simpler model, regularization, cross validation, early stopping, or ensembles.

### Q3. What is regularization?

**Answer:** A **penalty on model complexity** (usually on large coefficients) added to the training objective. It discourages the model from fitting noise, reducing overfitting. Its strength is controlled by a hyperparameter (`alpha`, or `C` — where a *smaller* `C` means *stronger* regularization).

### Q4. L1 vs L2

**Answer:** | | **L1 (Lasso)** | **L2 (Ridge)** |
|---|---|---|
| Penalty | Sum of \|coefficients\| | Sum of coefficients² |
| Effect | Can shrink coefficients **exactly to zero** | Shrinks coefficients **toward zero**, rarely zero |
| Use for | Feature selection, sparse models | Correlated features, keeping all features |

*Elastic Net* mixes both.

### Q5. Grid Search vs Random Search

**Answer:** | | **Grid Search** | **Random Search** |
|---|---|---|
| Tries | Every combination | A random sample of combinations |
| Speed | Slow as the grid grows | Faster (fixed `n_iter` trials) |
| Best for | Small search spaces | Large search spaces / limited time |

Grid search is thorough but expensive; random search often finds a nearly-as-good result with far fewer trials.

### Q6. What is Cross Validation?

**Answer:** Splitting the data into K folds, training on K-1 folds and validating on the remaining one, repeating K times, then **averaging the scores**. It gives a more reliable performance estimate than a single train/test split. Variants: Stratified K-Fold (imbalanced classes), Group K-Fold (grouped rows), Time Series Split (time-ordered data).

### Q7. Bagging vs Boosting

**Answer:** | | **Bagging** | **Boosting** |
|---|---|---|
| Training | Models trained **independently, in parallel** on bootstrap samples | Models trained **sequentially**, each fixing previous errors |
| Main effect | Reduces **variance** (overfitting) | Reduces **bias** (underfitting) |
| Example | Random Forest | AdaBoost, Gradient Boosting, XGBoost |

### Q8. Explain Random Forest

**Answer:** An ensemble of many decision trees. Each tree is trained on a **bootstrap sample** of the rows and considers only a **random subset of features** at each split. Predictions are combined by voting (classification) or averaging (regression). Averaging many different trees lowers variance, so it overfits far less than a single tree. It also provides feature importance and an out-of-bag (OOB) score.

### Q9. Explain Gradient Boosting

**Answer:** Builds many **small trees one after another**. Each new tree is trained to predict the **errors (residuals / gradient of the loss)** of the ensemble so far, and its output is added with a small weight (`learning_rate`). Key hyperparameters: `n_estimators`, `learning_rate`, `max_depth`, `subsample`. It is very accurate but can overfit if there are too many trees or they are too deep.

### Q10. What is XGBoost?

**Answer:** An optimized, regularized implementation of gradient boosting. It adds L1/L2 regularization on leaf weights, uses second-order gradient information, handles missing values natively, and trains fast through parallelism. It has a scikit-learn-compatible API (`XGBClassifier`). Install with `pip install xgboost`.

### Q11. What is class imbalance?

**Answer:** When one class is much rarer than the other (e.g., only ~11% of our orders are international). Models tend to favor the majority class. Ways to handle it: use the right metrics, **class weights**, **oversampling / undersampling / SMOTE**, and **threshold adjustment**.

### Q12. Why can accuracy be misleading?

**Answer:** On imbalanced data, a model that **always predicts the majority class** gets high accuracy (~89% here) while detecting **none** of the minority class. Use **precision, recall, F1, PR-AUC** instead.

### Q13. Precision vs Recall

**Answer:** - **Precision** = TP / (TP + FP): of everything predicted positive, how much was correct? (Avoids false alarms — e.g., spam filters.)
- **Recall** = TP / (TP + FN): of all actual positives, how many did we find? (Avoids misses — e.g., disease screening.)

They trade off: raising one usually lowers the other. **F1** is their harmonic mean.

### Q14. What is threshold tuning?

**Answer:** Classifiers output a probability; the label depends on a cutoff (default 0.5). **Threshold tuning** means choosing a different cutoff to trade precision against recall — e.g., lowering it to catch more positives. The threshold should be chosen on **validation / cross-validated data** (never the test set), guided by the business cost of false positives vs false negatives.

### Q15. What is model explainability?

**Answer:** Being able to understand **why** a model makes its predictions. **Global** explanations show overall drivers (coefficients, feature importance, permutation importance); **local** explanations explain a single prediction (SHAP, LIME). It builds trust, helps debugging, and is often required in regulated settings.

### Q16. What is error analysis?

**Answer:** Studying **where and why** a model makes mistakes: look at the confusion matrix, error rates by segment (country, price range, time), and examples of false positives/negatives. It reveals weaknesses and guides improvements (new features, more data, different threshold) better than a single score.

### Q17. How would you select the final model?

**Answer:** 1. Pick metrics that match the **business goal** (not just accuracy).
2. Build a **baseline**.
3. Compare several models using **cross validation**.
4. Check **generalization** (train vs validation gap) for overfitting.
5. **Tune** the best candidates and optimize the **threshold**.
6. Consider speed, complexity, interpretability and resource needs.
7. Confirm **once** on the untouched test set and pick the simplest model that meets the requirement — not simply the highest accuracy.

# Part B – Coding Exercises

## Setup (used by all exercises)
Target: is the order **international** (non-UK)? Features: quantity, price, total price, and time features. Metric: **ROC-AUC**.

In [1]:
import pandas as pd, numpy as np, time
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.inspection import permutation_importance
df = pd.read_csv('data.csv', encoding='latin1')
df = df.dropna(subset=['CustomerID'])
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)].sample(5000, random_state=42)
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']
df['Hour'] = df['InvoiceDate'].dt.hour
df['DayOfWeek'] = df['InvoiceDate'].dt.dayofweek
df['Month'] = df['InvoiceDate'].dt.month
X = df[['Quantity', 'UnitPrice', 'TotalPrice', 'Hour', 'DayOfWeek', 'Month']]
y = (df['Country'] != 'United Kingdom').astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

### E1. Cross Validation
Run 5-fold stratified cross validation for a Random Forest and report the scores, mean and standard deviation.

**Solution:**

In [4]:
scores = cross_val_score(RandomForestClassifier(n_estimators=100, random_state=42),
                         X_train, y_train, cv=skf, scoring='roc_auc')
print("Fold scores:", scores.round(3))
print("Mean:", scores.mean().round(3), "| Std:", scores.std().round(3))

Fold scores: [0.701 0.718 0.704 0.693 0.699]
Mean: 0.703 | Std: 0.008


### E2. Detect Overfitting
Train a Decision Tree with different `max_depth` values. Compare training AUC with cross-validated AUC. Which depths overfit?

**Solution:** (a large gap between train and CV AUC means overfitting — deeper trees show it clearly)

In [5]:
for d in [1, 3, 5, 10, None]:
    tree = DecisionTreeClassifier(max_depth=d, random_state=42).fit(X_train, y_train)
    train_auc = roc_auc_score(y_train, tree.predict_proba(X_train)[:, 1])
    cv_auc = cross_val_score(tree, X_train, y_train, cv=skf, scoring='roc_auc').mean()
    print(f"max_depth={d}: train={train_auc:.3f}, cv={cv_auc:.3f}, gap={train_auc - cv_auc:.3f}")

max_depth=1: train=0.669, cv=0.669, gap=0.000
max_depth=3: train=0.726, cv=0.703, gap=0.023
max_depth=5: train=0.756, cv=0.703, gap=0.053
max_depth=10: train=0.876, cv=0.653, gap=0.223
max_depth=None: train=1.000, cv=0.575, gap=0.425


### E3. Tune a Random Forest (manually)
Loop over `n_estimators` in [50, 100] and `max_depth` in [4, 8, None]. Use 3-fold CV AUC to find the best combination.

**Solution:**

In [6]:
best_score, best_params = 0, None
for n in [50, 100]:
    for d in [4, 8, None]:
        rf = RandomForestClassifier(n_estimators=n, max_depth=d, random_state=42)
        score = cross_val_score(rf, X_train, y_train, cv=3, scoring='roc_auc').mean()
        print(f"n_estimators={n}, max_depth={d}: {score:.3f}")
        if score > best_score:
            best_score, best_params = score, (n, d)
print("Best (n_estimators, max_depth):", best_params, "| CV AUC:", round(best_score, 3))

n_estimators=50, max_depth=4: 0.731
n_estimators=50, max_depth=8: 0.718
n_estimators=50, max_depth=None: 0.682
n_estimators=100, max_depth=4: 0.731
n_estimators=100, max_depth=8: 0.719
n_estimators=100, max_depth=None: 0.684
Best (n_estimators, max_depth): (100, 4) | CV AUC: 0.731


### E4. GridSearchCV
Use `GridSearchCV` to tune a Random Forest over `n_estimators`, `max_depth` and `min_samples_leaf`. Print the best parameters and best score, then evaluate on the test set.

**Solution:**

In [8]:
param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [4, 8],
    'min_samples_leaf': [1, 10],
}
grid = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=3, scoring='roc_auc')
grid.fit(X_train, y_train)
print("Best params:", grid.best_params_)
print("Best CV AUC:", round(grid.best_score_, 3))
print("Test AUC:   ", round(roc_auc_score(y_test, grid.predict_proba(X_test)[:, 1]), 3))

Best params: {'max_depth': 4, 'min_samples_leaf': 10, 'n_estimators': 100}
Best CV AUC: 0.734
Test AUC:    0.71


### E5. RandomizedSearchCV
Use `RandomizedSearchCV` (6 random trials) to tune a Gradient Boosting model. Print the best parameters and score.

**Solution:**

In [9]:
param_dist = {
    'n_estimators': [100, 200, 300],
    'max_depth': [2, 3, 4],
    'learning_rate': [0.03, 0.05, 0.1],
    'subsample': [0.7, 0.85, 1.0],
}
rand = RandomizedSearchCV(GradientBoostingClassifier(random_state=42), param_dist,
                          n_iter=6, cv=3, scoring='roc_auc', random_state=42)
rand.fit(X_train, y_train)
print("Best params:", rand.best_params_)
print("Best CV AUC:", round(rand.best_score_, 3))

Best params: {'subsample': 0.85, 'n_estimators': 200, 'max_depth': 2, 'learning_rate': 0.05}
Best CV AUC: 0.743


### E6. Compare Multiple Models
Compare Logistic Regression, Decision Tree (depth 5), Random Forest and Gradient Boosting on CV AUC, test AUC and training time.

**Solution:**

In [10]:
candidates = {
    'Logistic Regression': make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
}
rows = []
for name, model in candidates.items():
    start = time.time(); model.fit(X_train, y_train); fit_time = time.time() - start
    rows.append({
        'Model': name,
        'CV AUC': cross_val_score(model, X_train, y_train, cv=skf, scoring='roc_auc').mean(),
        'Test AUC': roc_auc_score(y_test, model.predict_proba(X_test)[:, 1]),
        'Training Time (s)': fit_time,
    })
pd.DataFrame(rows).round(3).sort_values('CV AUC', ascending=False)

,Model,CV AUC,Test AUC,Training Time (s)
3,Gradient Boosting,0.742,0.694,0.791
2,Random Forest,0.728,0.711,0.783
1,Decision Tree,0.703,0.697,0.015
0,Logistic Regression,0.619,0.616,0.060


### E7. Threshold Tuning
Using the Gradient Boosting model, compute precision, recall and F1 at thresholds 0.05, 0.1, 0.2, 0.3, 0.5. Which threshold gives the best F1?

**Solution:** (in practice, choose the threshold on validation/CV data — here we only illustrate the effect)

In [11]:
gb = candidates['Gradient Boosting']
probs = gb.predict_proba(X_test)[:, 1]
for t in [0.05, 0.1, 0.2, 0.3, 0.5]:
    pred = (probs >= t).astype(int)
    print(f"t={t}: precision={precision_score(y_test, pred, zero_division=0):.3f}, "
          f"recall={recall_score(y_test, pred, zero_division=0):.3f}, "
          f"F1={f1_score(y_test, pred, zero_division=0):.3f}")

t=0.05: precision=0.164, recall=0.927, F1=0.279
t=0.1: precision=0.159, recall=0.716, F1=0.260
t=0.2: precision=0.212, recall=0.284, F1=0.243
t=0.3: precision=0.324, recall=0.110, F1=0.164
t=0.5: precision=0.545, recall=0.055, F1=0.100


### E8. Model Explainability
Use **permutation importance** on the test set to find which features matter most to the Random Forest.

**Solution:** (a feature is important if shuffling it makes the AUC drop a lot)

In [12]:
rf = candidates['Random Forest']
result = permutation_importance(rf, X_test, y_test, scoring='roc_auc', n_repeats=5, random_state=42)
pd.Series(result.importances_mean, index=X.columns).sort_values(ascending=False).round(4)

TotalPrice    0.1101
Quantity      0.0620
Hour          0.0184
UnitPrice     0.0093
Month        -0.0003
DayOfWeek    -0.0051
dtype: float64

### E9. Error Analysis
For the Gradient Boosting model at threshold 0.15, show the confusion matrix and the recall for the 5 most common international countries.

**Solution:**

In [ ]:
pred = (probs >= 0.15).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
print(f"TN={tn}, FP={fp}, FN={fn}, TP={tp}")
errors = X_test.copy()
errors['y'] = y_test.values
errors['pred'] = pred
errors['Country'] = df.loc[X_test.index, 'Country']
intl = errors[errors['y'] == 1]
intl.groupby('Country')['pred'].agg(orders='count', recall='mean').sort_values('orders', ascending=False).head(5).round(2)